# FMT2 format-compliance test — google/gemma-3-4b-it on T4
**Development evidence only. Not research evidence and not pilot data.** Runs 12 AI-authored, unreviewed, permanently EXCLUDED fixture families (36 E/R/C prompts) through the unchanged Test 1 engine, frozen English instruction and inference recipe.
It measures how often correct-looking answers break the strict FINAL format, and reports scorer v1.0.0 beside the proposed secondary `numeric_literal_v1` (decision D019).
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Model download and loading take several minutes. At the end a ZIP downloads: send it to the team.
Never tune the prompt on these outputs and then reuse them as evidence; any prompt change is a new protocol version.


In [ ]:
import os, sys, json, base64, io, zipfile, hashlib, subprocess, traceback
MODEL_ID = 'google/gemma-3-4b-it'
if True:
    # Gated model: accept its licence on huggingface.co, then add a Colab secret named HF_TOKEN (key icon, left sidebar).
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as error:
        print('HF_TOKEN secret not available:', type(error).__name__, '— gated model download will fail')
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
PACKAGE = Path('/content/hinglish_fmt2') if Path('/content').exists() else Path.cwd() / 'hinglish_fmt2'
PACKAGE.mkdir(parents=True, exist_ok=True)
blob = base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest() == '0ca4a7b755a651dc3e0965b0886cc52dd321e956aa8543f85e9f97a72a121922'
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(PACKAGE)
sys.path.insert(0, str(PACKAGE))
from hinglish_pilot.engine import default_config, environment, read, write, run, utc
from hinglish_pilot.data import validate
from hinglish_pilot.report import analyze, export
from secondary_analysis.report_secondary import analyze as analyze_secondary
RUNS = PACKAGE / 'runs'
RUNS.mkdir(exist_ok=True)
STATUS = {'label': 'FMT2 FORMAT TEST — DEVELOPMENT EVIDENCE ONLY', 'started_at': utc(), 'stages': {}}
def note(stage, ok, error=None):
    STATUS['stages'][stage] = {'ok': ok, 'exception_class': type(error).__name__ if error else None}
    write(RUNS / 'fmt2_status.json', STATUS)
PARTS = [(n, read(PACKAGE / f'fmt2/{n}_dataset.json'), read(PACKAGE / f'fmt2/{n}_splits.json'))
         for n in ('fmt2-part-a', 'fmt2-part-b')]
for name, dataset, splits in PARTS:
    print(name, validate(dataset, splits, 'demo')['prompts'], 'prompts validated')


In [ ]:
GPU_READY = False
try:
    import torch
    assert torch.cuda.is_available(), 'Select a T4 GPU runtime'
    print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PACKAGE / 'requirements-colab.txt')], check=True)
    GPU_READY = True
    note('dependencies', True)
except Exception as error:
    note('dependencies', False, error); traceback.print_exc()
write(RUNS / 'fmt2_environment.json', environment())


In [ ]:
backend = None
CONFIG = default_config('demo', MODEL_ID)
if GPU_READY:
    try:
        from hinglish_pilot.hf_backend import resolve_config, HFBackend
        CONFIG = resolve_config(CONFIG)
        backend = HFBackend(CONFIG)
        write(RUNS / 'fmt2_model_load_report.json', backend.load_report)
        print('Model loaded on', backend.metadata['hardware'], 'in', round(backend.load_report['load_seconds']), 's')
        note('model_load', True)
    except Exception as error:
        note('model_load', False, error); traceback.print_exc()
write(RUNS / 'fmt2_config.json', CONFIG)


In [ ]:
RESULTS = []
if backend is not None:
    for name, dataset, splits in PARTS:
        try:
            progress = lambda done, total, item: print(f'{name} {done}/{total}: {item["rendering_id"]} {item["latency_seconds"]:.1f}s', flush=True)
            manifest = run(RUNS / name, dataset, splits, CONFIG, backend, progress=progress)
            analyze(RUNS / name)
            metrics, folder = analyze_secondary(RUNS / name)
            print((folder / 'secondary_summary.md').read_text())
            RESULTS.append((name, manifest['status']))
            note(name, manifest['status'] == 'COMPLETE')
        except Exception as error:
            note(name, False, error); traceback.print_exc()


## Download
Run this even if something failed: it packages whatever exists. Also keep the executed notebook (File → Download → Download .ipynb).


In [ ]:
STATUS['finished_at'] = utc()
write(RUNS / 'fmt2_status.json', STATUS)
archive = PACKAGE / ('FMT2_format_test_results' + ('' if MODEL_ID == 'Qwen/Qwen3-4B' else '_' + MODEL_ID.split('/')[-1]) + '.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for name, _, _ in PARTS:
        if (RUNS / name / 'run_manifest.json').exists():
            try:
                z.write(export(RUNS / name), f'{name}_results.zip')
            except Exception as error:
                note(f'{name}_export', False, error); traceback.print_exc()
    for path in sorted(RUNS.rglob('*')):
        if path.is_file() and ('secondary_diagnostic_' in str(path) or path.name.startswith('fmt2_')):
            z.write(path, str(path.relative_to(RUNS)))
print('Return:', archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Local: ZIP saved on disk.')
